In [0]:
# Install required library to read Excel files
%pip install openpyxl

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Using df_memb only — RFM requires known customer identity
df_memb = pd.read_parquet("/Volumes/workspace/default/retail_data/df_memb.parquet")

# DATASET INTEGRITY CHECK
# Validating transaction date boundaries before RFM calculation
# ---------------------------------------------------------------

# Find min and max transaction date for every month
monthly_date_range = df_memb.groupby(
    df_memb['InvoiceDate'].dt.to_period('M')
)['InvoiceDate'].agg(['min', 'max']).reset_index()

monthly_date_range.columns = ['YearMonth', 'First Transaction', 'Last Transaction']

print("TRANSACTION DATE RANGE PER MONTH")
print(monthly_date_range.to_string(index=False))


In [0]:
# Set analysis date — one day after the last transaction in the dataset
# This is used to calculate recency (how many days since last purchase)

# to calculate how many days ago each customer last purchased. 
# The last purchase date is "2011-12-09". 
# Using the day after avoids a zero recency score for the most recent customers.
analysis_date = pd.Timestamp('2011-12-10')

print(f"Data loaded — {len(df_memb):,} rows")

print(f"\nDataset starts: {df_memb['InvoiceDate'].min()}")
print(f"Dataset ends: {df_memb['InvoiceDate'].max()}")
print(f"Analysis date: {analysis_date.date()}")

In [0]:
# RFM CALCULATION
# For every customer we calculate:
    # Recency   — days since last purchase
    # Frequency — number of unique orders
    # TotalSpend — total amount spent
# -----------------------------------

rfm = df_memb.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda x: (analysis_date - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    TotalSpend=('TotalRevenue', 'sum')
).reset_index()

print("RFM TABLE — FIRST 10 CUSTOMERS")
print(rfm.head(10).to_string(index=False))

print(f"\nRFM SUMMARY STATISTICS")
print(rfm[['Recency', 'Frequency', 'TotalSpend']].describe().round(2))

In [0]:
# RFM SCORING
# Every customer gets a score of 1-5 for each
# R, F and TotalSpend dimension
# 5 = best, 1 = worst
#
# Recency:   lower days = better = higher score
# Frequency: more orders = better = higher score
# TotalSpend: more spend = better = higher score
# -----------------------------------------------

# We use pd.qcut to split customers into 5 equal groups
# qcut splits by quantile — each group has equal number of customers

# RECENCY SCORE
# Note: for recency we reverse the score (5=lowest days, 1=highest days)
# because buying recently is BETTER than buying a long time ago
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=5, labels=[5,4,3,2,1])

# FREQUENCY SCORE
# More orders = better = higher score
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5])

# TOTALSPEND SCORE
# More spend = better = higher score
rfm['S_Score'] = pd.qcut(rfm['TotalSpend'], q=5, labels=[1,2,3,4,5])

# COMBINE INTO ONE RFM SCORE
# Concatenate the three scores into one string e.g. "555" or "123"
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['S_Score'].astype(str)

# CALCULATE TOTAL SCORE (sum of all three)
rfm['Total_Score'] = rfm['R_Score'].astype(int) + rfm['F_Score'].astype(int) + rfm['S_Score'].astype(int)

print("RFM SCORES — FIRST 10 CUSTOMERS")
print(rfm[['Customer ID','Recency','Frequency','TotalSpend','R_Score','F_Score','S_Score','RFM_Score','Total_Score']].head(10).to_string(index=False))

In [0]:
# CUSTOMER SEGMENTATION
# Assigning every customer a meaningful label based on their RFM scores
# --------------------------------------------

def assign_segment(row):
    r = row['R_Score']
    f = row['F_Score']
    s = row['S_Score']
    total = row['Total_Score']

    if r >= 4 and f >= 4 and s >= 4:
        return 'Champion'
    elif r >= 3 and f >= 3 and s >= 3:
        return 'Loyal Customer'
    elif r >= 4 and f <= 2:
        return 'New Customer'
    elif r >= 3 and f >= 2 and s >= 2:
        return 'Potential Loyalist'
    elif r <= 2 and f >= 3 and s >= 3:
        return 'At Risk'
    elif r <= 2 and f <= 2 and s >= 3:
        return 'Cannot Lose Them'
    elif r <= 2 and f <= 2 and s <= 2:
        return 'Lost'
    else:
        return 'Needs Attention'

# Apply segment to every customer
rfm['Segment'] = rfm.apply(assign_segment, axis=1)

# Count customers per segment
segment_counts = rfm['Segment'].value_counts().reset_index()
segment_counts.columns = ['Segment', 'Customer Count']

# Revenue per segment
segment_revenue = rfm.groupby('Segment')['TotalSpend'].sum().reset_index()
segment_revenue.columns = ['Segment', 'Total Revenue']

# Merge both
segment_summary = segment_counts.merge(segment_revenue, on='Segment')

# Add revenue percentage
segment_summary['Revenue %'] = (
    segment_summary['Total Revenue'] / segment_summary['Total Revenue'].sum() * 100
).round(2)

# Add customer percentage
segment_summary['Customer %'] = (
    segment_summary['Customer Count'] / segment_summary['Customer Count'].sum() * 100
).round(2)

print("CUSTOMER SEGMENTS")
print(segment_summary[['Segment', 'Customer Count', 'Customer %', 'Total Revenue', 'Revenue %']].sort_values('Total Revenue', ascending=False).to_string(index=False))

In [0]:
# SEGMENT VISUALISATION
# Two charts side by side:
# 1. Customer count per segment
# 2. Revenue per segment
# ------------------------------

fig, axes = plt.subplots(1, 2, figsize=(25, 10))

# Sort by Total Revenue for consistent ordering
segment_sorted = segment_summary.sort_values('Total Revenue', ascending=True)

# --- CHART 1: CUSTOMER COUNT PER SEGMENT ---
axes[0].barh(
    segment_sorted['Segment'],
    segment_sorted['Customer Count'],
    color='#4C72B0'
)
axes[0].set_title('Customer Count by Segment', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Number of Customers', fontsize=11)

# Add percentage labels
for i, (count, pct) in enumerate(zip(segment_sorted['Customer Count'], segment_sorted['Customer %'])):
    axes[0].text(count + 10, i, f'{count} ({pct}%)', va='center', fontsize=9)

# --- CHART 2: REVENUE PER SEGMENT ---
axes[1].barh(
    segment_sorted['Segment'],
    segment_sorted['Total Revenue'],
    color='#2ecc71'
)
axes[1].set_title('Total Revenue by Segment', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Revenue (£)', fontsize=11)

# Add percentage labels
for i, (rev, pct) in enumerate(zip(segment_sorted['Total Revenue'], segment_sorted['Revenue %'])):
    axes[1].text(rev + 10000, i, f'£{rev:,.0f} ({pct}%)', va='center', fontsize=9)

plt.suptitle('Customer Segmentation Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [0]:
# RFM HEATMAP
# Shows average Recency, Frequency and TotalSpend for each segment in one view
# Helps understand what makes each segment different from the others
# -----------------------------------------------------------------------------

# Calculate average RFM values per segment
segment_rfm_avg = rfm.groupby('Segment').agg(
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_TotalSpend=('TotalSpend', 'mean')
).round(2)

# Normalise values between 0 and 1 for heatmap colouring
# This makes all three metrics comparable on the same scale
segment_rfm_norm['Avg_Recency'] = 1 - segment_rfm_norm['Avg_Recency']

# Plot heatmap
plt.figure(figsize=(20, 16))
sns.heatmap(
    segment_rfm_norm,
    annot=segment_rfm_avg,
    fmt='.1f',
    cmap='RdYlGn',
    linewidths=0.5,
    cbar=True
)

plt.title('Average RFM Values by Segment', fontsize=16, fontweight='bold')
plt.xlabel('RFM Metrics', fontsize=12)
plt.ylabel('Segment', fontsize=12)
plt.tight_layout()
plt.show()